In [4]:
import pandas as pd
import numpy as np
import numpy as np
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV

df = pd.read_csv('data.csv')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [5]:
df.head(6)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
5,9305-CDSKC,Female,0,No,No,8,Yes,Yes,Fiber optic,No,...,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.5,Yes


In [6]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


In [7]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [8]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.loc[df['TotalCharges'].isna(), 'TotalCharges'] = df['TotalCharges'].mean()

In [9]:
df.loc[df['tenure'] == 0, 'TotalCharges']

488     2283.300441
753     2283.300441
936     2283.300441
1082    2283.300441
1340    2283.300441
3331    2283.300441
3826    2283.300441
4380    2283.300441
5218    2283.300441
6670    2283.300441
6754    2283.300441
Name: TotalCharges, dtype: float64

In [10]:
df = pd.get_dummies(df, columns=['Contract','InternetService','MultipleLines','PaymentMethod'], dtype=int)

In [11]:
cols_to_encode = ['StreamingTV', 'StreamingMovies','OnlineSecurity','TechSupport','OnlineBackup','DeviceProtection']
mapping = {'Yes': 1, 'No': 0, 'No internet service': 0}
for col in cols_to_encode:
    df[col] = df[col].map(mapping)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0})
cols_to_encode = ['Dependents', 'PhoneService','Partner','PaperlessBilling']
mapping = {'Yes': 1, 'No': 0, 'No internet service': 0}
for col in cols_to_encode:
    df[col] = df[col].map(mapping)

df.drop(columns=['customerID','OnlineBackup','DeviceProtection',], inplace=True)
df.shape

(7043, 27)

In [12]:
df.nunique()

gender                                        2
SeniorCitizen                                 2
Partner                                       2
Dependents                                    2
tenure                                       73
PhoneService                                  2
OnlineSecurity                                2
TechSupport                                   2
StreamingTV                                   2
StreamingMovies                               2
PaperlessBilling                              2
MonthlyCharges                             1585
TotalCharges                               6531
Churn                                         2
Contract_Month-to-month                       2
Contract_One year                             2
Contract_Two year                             2
InternetService_DSL                           2
InternetService_Fiber optic                   2
InternetService_No                            2
MultipleLines_No                        

In [13]:
df.info()
df.drop([])

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 27 columns):
 #   Column                                   Non-Null Count  Dtype  
---  ------                                   --------------  -----  
 0   gender                                   7043 non-null   int64  
 1   SeniorCitizen                            7043 non-null   int64  
 2   Partner                                  7043 non-null   int64  
 3   Dependents                               7043 non-null   int64  
 4   tenure                                   7043 non-null   int64  
 5   PhoneService                             7043 non-null   int64  
 6   OnlineSecurity                           7043 non-null   int64  
 7   TechSupport                              7043 non-null   int64  
 8   StreamingTV                              7043 non-null   int64  
 9   StreamingMovies                          7043 non-null   int64  
 10  PaperlessBilling                         7043 non-null   in

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,TechSupport,StreamingTV,StreamingMovies,...,InternetService_DSL,InternetService_Fiber optic,InternetService_No,MultipleLines_No,MultipleLines_No phone service,MultipleLines_Yes,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,0,1,0,1,0,0,0,0,0,...,1,0,0,0,1,0,0,0,1,0
1,1,0,0,0,34,1,1,0,0,0,...,1,0,0,1,0,0,0,0,0,1
2,1,0,0,0,2,1,1,0,0,0,...,1,0,0,1,0,0,0,0,0,1
3,1,0,0,0,45,0,1,1,0,0,...,1,0,0,0,1,0,1,0,0,0
4,0,0,0,0,2,1,0,0,0,0,...,0,1,0,1,0,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,1,0,1,1,24,1,1,1,1,1,...,1,0,0,0,0,1,0,0,0,1
7039,0,0,1,1,72,1,0,0,1,1,...,0,1,0,0,0,1,0,1,0,0
7040,0,0,1,1,11,0,1,0,0,0,...,1,0,0,0,1,0,0,0,1,0
7041,1,1,1,0,4,1,0,0,0,0,...,0,1,0,0,0,1,0,0,0,1


In [ ]:
y = df['Churn']
X = df.drop('Churn', axis=1)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)
model = RandomForestClassifier(n_estimators=100,max_depth=10,class_weight='balanced',random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'class_weight': ['balanced', None]  
}
grid_search = GridSearchCV(RandomForestClassifier(random_state=42), param_grid=params, cv=5, scoring='recall')
grid_search.fit(X_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestC...ndom_state=42)
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'class_weight': ['balanced', None], 'max_depth': [10, 20, ...], 'n_estimators': [100, 200, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'recall'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, defa

In [15]:
print(accuracy_score(y_test, y_pred))
features = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
})
print(features.sort_values(by='importance', ascending=False))
print(model.feature_importances_)

0.7601135557132718
                                    feature  importance
13                  Contract_Month-to-month    0.143617
4                                    tenure    0.143582
12                             TotalCharges    0.131526
11                           MonthlyCharges    0.121081
15                        Contract_Two year    0.077851
17              InternetService_Fiber optic    0.062346
24           PaymentMethod_Electronic check    0.040034
18                       InternetService_No    0.033142
14                        Contract_One year    0.026451
10                         PaperlessBilling    0.023909
6                            OnlineSecurity    0.022642
16                      InternetService_DSL    0.020509
7                               TechSupport    0.016412
0                                    gender    0.015104
9                           StreamingMovies    0.014236
2                                   Partner    0.013833
3                            

In [16]:
y_pred = model.predict(X_test)
classification_report(y_test, y_pred)


'              precision    recall  f1-score   support\n\n           0       0.90      0.76      0.82      1035\n           1       0.53      0.77      0.63       374\n\n    accuracy                           0.76      1409\n   macro avg       0.72      0.76      0.73      1409\nweighted avg       0.80      0.76      0.77      1409\n'

In [19]:
zx = model.predict_proba(X_test)
zx

array([[0.98006262, 0.01993738],
       [0.09103485, 0.90896515],
       [0.88642017, 0.11357983],
       ...,
       [0.77531723, 0.22468277],
       [0.90515035, 0.09484965],
       [0.95917699, 0.04082301]], shape=(1409, 2))